In [2]:
import cv2
import numpy as np
import os

input_folder = "enhanced_frames"
output_folder = "output_alligator_cracks"

os.makedirs(output_folder, exist_ok=True)

def road_mask(gray):

    h, w = gray.shape

    mask = np.zeros((h, w), dtype=np.uint8)

    # alligator cracks occur in central wheel path zone
    mask[int(h*0.40):h, int(w*0.15):int(w*0.85)] = 255

    return mask

def detect_alligator(gray, output, mask):

    roi = cv2.bitwise_and(gray, gray, mask=mask)

    blur = cv2.GaussianBlur(roi, (7,7), 0)

    edges = cv2.Canny(blur, 50, 140)

    h, w = edges.shape

    block_size = 50
    step = 25

    crack_map = np.zeros_like(edges)

    for y in range(0, h - block_size, step):
        for x in range(0, w - block_size, step):

            region = edges[y:y+block_size, x:x+block_size]

            density = np.mean(region > 0)

            # alligator cracks = dense irregular mesh patterns
            if density > 0.18:
                crack_map[y:y+block_size, x:x+block_size] = 255

    kernel = np.ones((21,21), np.uint8)

    crack_map = cv2.morphologyEx(crack_map, cv2.MORPH_CLOSE, kernel, iterations=2)
    crack_map = cv2.morphologyEx(crack_map, cv2.MORPH_OPEN, np.ones((7,7), np.uint8))

    contours, _ = cv2.findContours(crack_map, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    count = 0

    for c in contours:

        area = cv2.contourArea(c)

        if area < 4000:
            continue

        x, y, w, h = cv2.boundingRect(c)

        aspect = w / (h + 1e-5)

        # alligator cracks = compact network-like regions
        if aspect > 3.5 or aspect < 0.3:
            continue

        roi_local = gray[y:y+h, x:x+w]

        std = np.std(roi_local)
        mean = np.mean(roi_local)

        if std < 15:
            continue

        if mean > 235:
            continue

        cv2.rectangle(output, (x,y), (x+w,y+h), (0,255,0), 3)

        cv2.putText(
            output,
            "Alligator Crack",
            (x, y-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,255,0),
            2
        )

        count += 1

    return count


files = sorted(os.listdir(input_folder))

for file in files:

    if not file.lower().endswith(('.png','.jpg','.jpeg')):
        continue

    path = os.path.join(input_folder, file)

    frame = cv2.imread(path)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    output = frame.copy()

    mask = road_mask(gray)

    count = detect_alligator(gray, output, mask)

    cv2.putText(
        output,
        f"Alligator Cracks: {count}",
        (20,40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0,255,0),
        2
    )

    cv2.imwrite(os.path.join(output_folder, file), output)

    print(f"{file} → {count} alligator cracks")

print("\n ALLIGATOR CRACK DETECTION COMPLETE")

AlligatorCracking_frame_0.jpg → 1 alligator cracks
AlligatorCracking_frame_20.jpg → 1 alligator cracks
AlligatorCracking_frame_35.jpg → 1 alligator cracks
AlligatorCracking_frame_4.jpg → 1 alligator cracks
AlligatorCracking_frame_51.jpg → 1 alligator cracks
AlligatorCracking_frame_9.jpg → 1 alligator cracks
EdgeCracking3_frame_72.jpg → 1 alligator cracks
EdgeCracking4_frame_123.jpg → 1 alligator cracks
EdgeCracking4_frame_129.jpg → 1 alligator cracks
EdgeCracking4_frame_130.jpg → 1 alligator cracks
EdgeCracking4_frame_133.jpg → 1 alligator cracks
EdgeCracking4_frame_144.jpg → 1 alligator cracks
EdgeCracking4_frame_154.jpg → 1 alligator cracks
EdgeCracking9_frame_185.jpg → 1 alligator cracks
EdgeCracking9_frame_202.jpg → 1 alligator cracks
EdgeCracking9_frame_206.jpg → 1 alligator cracks
EdgeCracking9_frame_217.jpg → 1 alligator cracks
RoadPotholes2_frame_522.jpg → 1 alligator cracks
RoadPotholes3_frame_575.jpg → 1 alligator cracks
RoadPotholes3_frame_578.jpg → 1 alligator cracks
RoadPo